## Masking out continuations that do not fit generation goals

We have a list of words that are forbidden, and another list of words that are brand words.
Choose continuations that do not use the forbidden words, and maximize the use of brand words.

In [1]:
#%pip install --upgrade --quiet transformers torch fbgemm-gpu accelerate

In [21]:
# CHANGE this to the Llama model for which you have applied for access via Hugging Face
# See: https://www.llama.com/docs/getting-the-models/hugging-face/
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

import os
# from dotenv import load_dotenv
# load_dotenv("../keys.env")
# assert os.environ["HF_TOKEN"][:2] == "hf",\
       # "Please sign up for access to the specific Llama model via HuggingFace and provide access token in keys.env file"

## Word lists

Set up the word lists.

"Banned words" from https://channelkey.com/amazon-content-seo-and-optimization/400-restricted-amazon-keywords-the-most-comprehensive-list-youll-ever-need/

In [22]:
# From https://channelkey.com/amazon-content-seo-and-optimization/400-restricted-amazon-keywords-the-most-comprehensive-list-youll-ever-need/
with open("./data/banned_phrases.txt") as ifp:
    banned_phrases = [line.strip().lower() for line in ifp.readlines()]
banned_phrases[100:105]

['decomposable', 'definitive', 'degradable', 'dementia', 'depression']

In [23]:
# Based on https://marketkeep.com/seo-keywords-for-nutrition/
with open("./data/desired_phrases.txt") as ifp:
    desired_phrases = [line.strip().lower() for line in ifp.readlines()]
desired_phrases[:5]

['nutrition', 'calorie deficit', 'diet', 'protein shake', 'paleo diet']

In [24]:
# makes unique
banned_phrases = set(banned_phrases)
desired_phrases = set(desired_phrases)

## Zero-shot generation

Without any logits processing

In [25]:
from transformers import pipeline

pipe = pipeline(
    task="text-generation", 
    model=MODEL_ID,
    use_fast=True,
    kwargs={
        "return_full_text": False,
    },
    model_kwargs={}
)

Loading weights: 100%|██████████| 254/254 [00:00<00:00, 1696.77it/s]


In [8]:
pipe

TextGenerationPipeline: {'model': 'LlamaForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}

In [26]:
def generate_product_description(item: str) -> str:
    system_prompt = f"""
        You are a product marketer for a company that makes nutrition supplements.
        Balance your product descriptions to attract customers, optimize SEO, and
        stay within accurate advertising guidelines.
        Product descriptions have to be 3-5 sentences.
        Provide only the product description with no preamble.
    """
    user_prompt = f"""
        Write a product description for a {item}.
    """

    input_message = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}   
    ]
    
    results = pipe(input_message, 
                   max_new_tokens=512)
    return results[0]['generated_text'][-1]['content'].strip()

prod = generate_product_description("protein drink")
print(prod)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_ou

Unleash your full potential with our high-quality protein drink, designed to support muscle growth, recovery, and overall well-being. With 20 grams of protein per serving, our drink is perfect for post-workout recovery, athletes, and fitness enthusiasts. Made with natural ingredients and no artificial flavors or sweeteners, our protein drink is a delicious and healthy way to fuel your active lifestyle.


In [27]:
import numpy as np

def evaluate(descr: str, positives, negatives) -> int:
    # go through and count the number of desired phrases and banned phrases
    descr = descr.lower()
    num_positive = np.sum([1 for phrase in positives if phrase in descr])
    num_negative = np.sum([1 for phrase in negatives if phrase in descr])
    return int(num_positive - num_negative)

def evaluate_verbose(descr: str, positives, negatives) -> int:
    # go through and count the number of desired phrases and banned phrases
    descr = descr.lower()
    
    num_positive = num_negative = 0
    for phrase in positives:
        if phrase in descr:
            num_positive += 1
            print(f"Good: {phrase}")
    for phrase in negatives:
        if phrase in descr:
            num_negative += 1
            print(f"Bad: {phrase}")
    print(f"Good: {num_positive}   Bad: {num_negative}")
    return num_positive - num_negative

evaluate_verbose(prod, desired_phrases, banned_phrases)

Good: healthy
Bad: perfect for
Bad: perfect
Bad: growth
Bad: heal
Bad: quality
Bad: natural
Good: 1   Bad: 6


-5

In [11]:
evaluate(prod, desired_phrases, banned_phrases)

-2

## Use logits processing to enhance the product description

We'll use the same prompt, but use logits processing to upvote/downvote specific words

In [ ]:
import torch
import numpy as np
from transformers.generation.logits_process import (
    LogitsProcessor,
    LOGITS_PROCESSOR_INPUTS_DOCSTRING,
)
from transformers.utils import add_start_docstrings
from transformers import AutoTokenizer, AutoModelForCausalLM
class BrandLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, positives, negatives, debug = True):
        self.tokenizer = tokenizer
        self.positives = positives
        self.negatives = negatives
        self.debug = debug
        self.history = []
        self.step = 0
    @add_start_docstrings(LOGITS_PROCESSOR_INPUTS_DOCSTRING)
    def __call__(
        self, input_ids: torch.LongTensor, input_logits: torch.FloatTensor
    ) -> torch.FloatTensor:
        output_logits = input_logits.clone()
            
        num_matches = [0] * len(input_ids)
        for idx, seq in enumerate(input_ids):
            # 여기서 말하는 seq는 빔 서치의 가능성 중에 하나임
            # decode the sequence
            decoded = self.tokenizer.decode(seq)
            num_matches[idx] = evaluate(decoded, self.positives, self.negatives)
        max_matches = np.max(num_matches)
          
        # logits goes from -inf to zero.  Mask out the non-max sequences; torch doesn't like it to be -np.inf
        for idx in range(len(input_ids)):
            if num_matches[idx] != max_matches:
                output_logits[idx] = -10000

        if self.debug:
            self.history.append({
                "step": self.step,
                "input_ids": input_ids.detach().cpu(),
                "num_matches": num_matches,
                "max_matches": int(max_matches),

                # 로그는 무겁기 때문에 보통 축소해서 저장
                "topk_before": torch.topk(input_logits, 5, dim=-1).values.detach().cpu(),
                "topk_after": torch.topk(output_logits, 5, dim=-1).values.detach().cpu(),

                # argmax token 추적
                "chosen_token_before": torch.argmax(input_logits, dim=-1).detach().cpu(),
                "chosen_token_after": torch.argmax(output_logits, dim=-1).detach().cpu(),
            })
            self.step += 1
                  
        return output_logits

In [31]:
def generate_product_description_v2(item: str) -> str:
    system_prompt = f"""
        You are a product marketer for a company that makes nutrition supplements.
        Balance your product descriptions to attract customers, optimize SEO, and
        stay within accurate advertising guidelines.
        Product descriptions have to be 3-5 sentences.
        Provide only the product description with no preamble.
    """
    user_prompt = f"""
        Write a product description for a {item}.
    """
    
    input_message = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}   
    ]
    
    # alliterate on the first letter of the animal. So, donkey would be D
    brand_processor = BrandLogitsProcessor(pipe.tokenizer, desired_phrases, banned_phrases)
    
    results = pipe(input_message, 
                   max_new_tokens=512,
                   do_sample=True,
                   temperature=0.8,
                   num_beams=10,
                   use_cache=True, # default is True
                   logits_processor=[brand_processor])

    return results[0]['generated_text'][-1]['content'].strip(), brand_processor.history

prod,logs = generate_product_description_v2("protein drink")
print(prod)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"Fuel your active lifestyle with our premium protein drink, packed with 20 grams of whey protein, 10 grams of branched-chain amino acids (BCAAs), and essential vitamins and minerals to support muscle recovery and overall well-being. Our unique blend of whey protein isolate and micellar casein provides a sustained release of nutrients, helping to build and repair muscle tissue. With no artificial flavors or sweeteners, our protein drink is a guilt-free way to support your fitness goals. Enjoy the taste of a refreshing beverage while nourishing your body with the nutrients it needs to thrive."


In [12]:
evaluate_verbose(prod, desired_phrases, banned_phrases)

Good: nutrients
Good: whey protein
Good: whey
Good: 3   Bad: 0


3

In [39]:
logs[2]["input_ids"].shape

torch.Size([10, 102])

In [41]:
logs[1]

{'step': 1,
 'input_ids': tensor([[128000, 128006,   9125,  ..., 128007,    271,      1],
         [128000, 128006,   9125,  ..., 128007,    271,   1090],
         [128000, 128006,   9125,  ..., 128007,    271,   1844],
         ...,
         [128000, 128006,   9125,  ..., 128007,    271,  71999],
         [128000, 128006,   9125,  ..., 128007,    271,      0],
         [128000, 128006,   9125,  ..., 128007,    271,      2]]),
 'num_matches': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'max_matches': 0,
 'topk_before': tensor([[-1.4978e+00, -1.8728e+00, -1.8728e+00, -2.3728e+00, -2.3728e+00],
         [-5.8675e-02, -2.9337e+00, -5.9337e+00, -7.5587e+00, -8.5587e+00],
         [-6.5818e-03, -5.7566e+00, -7.7566e+00, -7.8191e+00, -8.2566e+00],
         [-1.0230e-02, -4.8852e+00, -7.0102e+00, -7.0102e+00, -7.8852e+00],
         [-1.7827e+00, -1.9702e+00, -2.3452e+00, -2.8452e+00, -2.8452e+00],
         [-1.0374e-02, -5.8229e+00, -6.6979e+00, -6.8229e+00, -7.0729e+00],
         [-4.5446e-01, -1.4545e